# fedOPD · build runbook — forecaster + federated loop

Successor to the EDA (closed at tag `eda-v2`). Turns the frozen 125-trust panel into trained
federated forecasters with an honest verdict against the **F-27 bar (naive-t-1 ~12.6% sMAPE)**.

**Conventions (see `docs/BUILD_LOG.md`, `docs/OPS.md`):**
- **Notebook-only** build, sectioned by step (owner instruction — overrides the runbook's modules+CLI
  proposal, recorded as D-06). Code here is AI-authored under the owner's explicit override of §12.
- **Device: CPU by default.** CUDA is used only if >20 GB is free (checked read-only so the shared,
  busy A100 is never disturbed). Whole grid ~7 TFLOP = CPU-minutes.
- Steps run in order; each ends in a **GATE**. A red gate STOPS the line.
- Findings are `R-xx`; decisions continue from `D-06`. Every run writes `results/*.json`.

## Step 0 · Preflight
Verify the inherited canonical table (D-01), pick the device without disturbing the GPU, cap threads,
and stamp the environment. **GATE:** row/month/column asserts pass and `results/env.json` is written.

In [1]:
# Step 0 - preflight: env, device policy, inherited canonical table.
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
import sys, json, hashlib, platform, subprocess, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd, torch
torch.set_num_threads(2)

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "CLAUDE.md").exists())
RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)

# Device: use CUDA only if >20 GB is free (checked read-only via nvidia-smi, so we never disturb the
# shared A100), else CPU. The whole grid is ~7 TFLOP -> CPU-minutes, so CPU blocks nothing.
def gpu_free_gb():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=5)
        return int(r.stdout.strip().splitlines()[0]) / 1024
    except Exception:
        return None
_free = gpu_free_gb()
DEVICE = "cuda" if (_free is not None and _free > 20 and torch.cuda.is_available()) else "cpu"
print(f"device: {DEVICE}  (gpu free: {None if _free is None else round(_free, 1)} GB)")

# Inherited canonical table (D-01) - the ONLY input; never re-read the raw CSVs.
canon = pd.read_parquet(ROOT / "reports" / "tables" / "rtt_canonical.parquet")
EXPECT_COLS = ["date", "branch", "branch_name", "icb", "dept", "dept_name", "demand"]
assert len(canon) == 227344, f"row count {len(canon)} != 227344 - EDA/build disagree, STOP"
assert canon.date.nunique() == 96, "month count != 96"
assert list(canon.columns) == EXPECT_COLS, f"columns {list(canon.columns)}"
print(f"canonical OK: {len(canon):,} rows | {canon.date.nunique()} months "
      f"{canon.date.min().date()}..{canon.date.max().date()}")

def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip()
    except Exception:
        return "?"

env = {"when": dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "host": platform.node(),
       "git": git_sha(), "python": sys.version.split()[0], "torch": torch.__version__,
       "numpy": np.__version__, "pandas": pd.__version__, "device": DEVICE,
       "gpu_free_gb": None if _free is None else round(_free, 1),
       "cpu_count": os.cpu_count(), "omp_threads": os.environ.get("OMP_NUM_THREADS"),
       "rows": int(len(canon)), "months": int(canon.date.nunique())}
(RESULTS / "env.json").write_text(json.dumps(env, indent=2))
print("GATE Step 0: PASS - asserts pass, env stamped ->", (RESULTS / "env.json"))

device: cpu  (gpu free: 6.6 GB)
canonical OK: 227,344 rows | 96 months 2018-04-01..2026-03-01
GATE Step 0: PASS - asserts pass, env stamped -> /home/garima/fedOPD/results/env.json


## Step 1 · Freeze the panel, roster, and eligible series
Apply D-03 (125-trust panel) and the F-28 roster decision (18 window-complete departments, excluding
the X 'Other' family - **D-07, pending supervisor Q5**). Filter to series that can yield >=24 train
windows and >=1 test window under window W=12 (D-02 validity). Freeze `results/panel.json` (hashed);
everything downstream reads this file. **GATE:** 125 trusts, series+drops logged, panel.json written.

In [2]:
# Step 1 - freeze panel, roster, and eval / train-only / dropped series -> results/panel.json
from collections import Counter
months = pd.Index(sorted(canon.date.unique()))
TRAIN_END = pd.Timestamp("2025-03-01"); W = 12; MIN_TRAIN_WINDOWS = 24
DEPTS_MODE = "stable18"; MIN_COVERAGE = 0.5                 # supervisor Q5 roster / Q4 threshold (parametrised)

pm = canon.groupby("branch").date.nunique()
panel = sorted(pm.index[pm == 96]); pset = set(panel)
variants = {"ge96_default": int((pm==96).sum()), "ge90": int((pm>=90).sum()), "ge84": int((pm>=84).sum())}
assert len(panel) == 125, f"panel {len(panel)} != 125"

# Roster (F-28): stable-18 = depts present all 96 months (excludes X01 36mo, X02-X06 60mo). D-07 default.
dept_months = canon.groupby("dept").date.nunique()
stable18 = sorted(dept_months.index[dept_months == 96])
roster_full = {"stable18": stable18, "all24": sorted(canon.dept.unique()),
               "no_x": sorted(d for d in canon.dept.unique() if not str(d).startswith("X"))}
cov = canon[canon.branch.isin(pset)].groupby("dept").branch.nunique() / len(panel)   # coverage among panel
roster = [d for d in roster_full[DEPTS_MODE] if cov.get(d, 0) >= MIN_COVERAGE]
roster_dropped = {d: round(float(cov.get(d, 0)), 3) for d in roster_full[DEPTS_MODE] if cov.get(d, 0) < MIN_COVERAGE}

# Window validity (D-02): window at target t valid iff months[t-W:t] present AND month t present.
tr_idx = [i for i, m in enumerate(months) if i >= W and m <= TRAIN_END]
te_idx = [i for i, m in enumerate(months) if m > TRAIN_END]
pres = (canon.assign(one=1).pivot_table("one", ["branch","dept"], "date", fill_value=0)
        .reindex(columns=months, fill_value=0).astype(bool))

# Three sets. A series with >=24 train windows but NO valid test window is NOT simply dropped: if it is
# still active (reports in a 2025-04+ month = interior gap) it is kept TRAIN-ONLY (it carries gradient
# signal to the shared model but cannot be evaluated); if it went stale (no report after 2025-03) it is
# dropped. Every method then trains on the identical set (eval + train-only) so comparisons hold.
eval_series, train_only, drops = [], [], []
for (b, d), row in pres.iterrows():
    if b not in pset or d not in roster:
        continue
    v = row.values
    tr = sum(bool(v[i]) and bool(v[i-W:i].all()) for i in tr_idx)
    te = sum(bool(v[i]) and bool(v[i-W:i].all()) for i in te_idx)
    if tr < MIN_TRAIN_WINDOWS:              drops.append((b, d, "few-train-windows"))
    elif te >= 1:                           eval_series.append([b, d, int(tr), int(te)])
    elif bool(v[te_idx].any()):             train_only.append([b, d, int(tr)])     # active interior-gap
    else:                                   drops.append((b, d, "stale-closed"))
reasons = Counter(r for _, _, r in drops)
train_series = [[b, d] for b, d, *_ in eval_series] + [[b, d] for b, d, _ in train_only]
client_trusts = sorted({b for b, d in train_series})
print(f"panel {len(panel)} trusts | roster {len(roster)}/{len(roster_full[DEPTS_MODE])} depts "
      f"| dropped-by-coverage {roster_dropped}")
print(f"eval {len(eval_series)} + train-only {len(train_only)} = {len(train_series)} training series "
      f"| {len(client_trusts)} client trusts | drops {dict(reasons)}")

panel_obj = {"created": env["when"], "git": env["git"], "panel_trusts": panel, "panel_variants": variants,
             "depts_mode": DEPTS_MODE, "roster": roster, "roster_dropped": roster_dropped,
             "min_coverage": MIN_COVERAGE, "window": W, "min_train_windows": MIN_TRAIN_WINDOWS,
             "train_end": str(TRAIN_END.date()), "test_months": [str(months[i].date()) for i in te_idx],
             "eval_series": eval_series, "train_only_series": train_only,
             "n_eval_series": len(eval_series), "n_train_series": len(train_series),
             "client_trusts": client_trusts, "n_clients": len(client_trusts), "drop_reasons": dict(reasons)}
panel_obj["hash"] = hashlib.sha256(json.dumps(panel_obj, sort_keys=True).encode()).hexdigest()[:16]
(RESULTS / "panel.json").write_text(json.dumps(panel_obj, indent=2))
ok = len(panel) == 125 and len(train_series) > 0 and (RESULTS / "panel.json").exists()
print("panel.json hash:", panel_obj["hash"]); print("GATE Step 1:", "PASS" if ok else "FAIL")

panel 125 trusts | roster 17/18 depts | dropped-by-coverage {'C_150': 0.36}
eval 1528 + train-only 16 = 1544 training series | 117 client trusts | drops {'few-train-windows': 137, 'stale-closed': 18}
panel.json hash: 7a4d1af76b21d143
GATE Step 1: PASS


## Step 2 · Build the datasets
Per series: reindex to month-start, window W=12 (horizon 1). **Features (33 = 12 scaled lags + month
sin/cos + COVID flag + 2023-strike flag + 17 dept one-hots).** Per-series scalers fitted on TRAIN months
only (≤2025-03), **excluding outlier-flagged months** (D-05); std floored at 1e-6. A window is used only
if all 12 lags and the target are present (D-02) — no imputation. Training set = eval + train-only series;
evaluation set = eval series' test windows. **GATE: five leakage asserts must pass.**

In [3]:
# Step 2 - windows + per-series scalers (TRAIN-only, excl. outlier-flagged months) -> results/dataset_stats.json
reg = pd.read_csv(ROOT/"reports"/"tables"/"07_outlier_register.csv", parse_dates=["date"])
flagged = set(zip(reg.branch, reg.dept, reg.date))         # months excluded from scaler fitting (D-05)

dem = canon.pivot_table("demand", ["branch","dept"], "date").reindex(columns=months)   # NaN where absent
mo = np.array([m.month for m in months])
MSIN = np.sin(2*np.pi*mo/12); MCOS = np.cos(2*np.pi*mo/12)
COVID  = ((months>=pd.Timestamp("2020-03-01"))&(months<=pd.Timestamp("2021-03-01"))).astype(float)
STRIKE = ((months>=pd.Timestamp("2022-12-01"))&(months<=pd.Timestamp("2023-12-01"))).astype(float)
ridx = {d:i for i,d in enumerate(roster)}; NR = len(roster)
N_FEATURES = W + 4 + NR
tr_targets = [i for i in range(W,len(months)) if months[i]<=TRAIN_END]
te_targets = [i for i in range(W,len(months)) if months[i]>TRAIN_END]

def fit_scaler(b, d):                                      # per-series mean/std, TRAIN only, excl. flagged
    row = dem.loc[(b,d)].values
    keep = [row[i] for i in range(len(months))
            if months[i] <= TRAIN_END and not np.isnan(row[i]) and (b,d,months[i]) not in flagged]
    if not keep: return 0.0, 1.0
    a = np.array(keep); return float(a.mean()), max(float(a.std()), 1e-6)

def make_row(row, t, mean, sd, oh):
    return np.concatenate([(row[t-W:t]-mean)/sd, [MSIN[t], MCOS[t], COVID[t], STRIKE[t]], oh])

scalers = {}
Xtr, ytr, tr_owner = [], [], []
Xte, yte, te_meta = [], [], []
for b, d in ([tuple(s[:2]) for s in eval_series] + [tuple(s[:2]) for s in train_only]):
    row = dem.loc[(b,d)].values; mean, sd = fit_scaler(b,d); scalers[(b,d)] = (mean, sd)
    oh = np.zeros(NR); oh[ridx[d]] = 1.0
    for t in tr_targets:
        if np.isnan(row[t]) or np.isnan(row[t-W:t]).any(): continue
        Xtr.append(make_row(row,t,mean,sd,oh)); ytr.append((row[t]-mean)/sd); tr_owner.append(b)
for s in eval_series:
    b, d = s[0], s[1]; row = dem.loc[(b,d)].values; mean, sd = scalers[(b,d)]
    oh = np.zeros(NR); oh[ridx[d]] = 1.0
    for t in te_targets:
        if np.isnan(row[t]) or np.isnan(row[t-W:t]).any(): continue
        Xte.append(make_row(row,t,mean,sd,oh)); yte.append((row[t]-mean)/sd)
        te_meta.append((b, d, str(months[t].date()), mean, sd, float(row[t])))
Xtr, ytr = np.asarray(Xtr, np.float32), np.asarray(ytr, np.float32)
Xte, yte = np.asarray(Xte, np.float32), np.asarray(yte, np.float32)
te_df = pd.DataFrame(te_meta, columns=["branch","dept","month","mean","sd","actual"])

# ---- GATE: five leakage asserts ----
tmonths = pd.to_datetime(te_df.month)
assert bool(((tmonths>=pd.Timestamp("2025-04-01"))&(tmonths<=pd.Timestamp("2026-03-01"))).all()), "test month out of range"
assert not (np.isnan(Xtr).any() or np.isnan(ytr).any() or np.isnan(Xte).any() or np.isnan(yte).any()), "NaN in X/y"
tr_trusts, te_trusts = set(tr_owner), set(te_df.branch)
assert te_trusts.issubset(tr_trusts) and len(te_trusts) > 0, "a test trust does not train"
for (b,d) in scalers:                                      # scaler fit never touched a test month
    row = dem.loc[(b,d)].values
    fit_months = [months[i] for i in range(len(months))
                  if months[i] <= TRAIN_END and not np.isnan(row[i]) and (b,d,months[i]) not in flagged]
    assert (not fit_months) or max(fit_months) <= TRAIN_END, f"scaler leak {b} {d}"
assert len(ytr) > 0 and len(yte) > 0

stats = {"n_features": N_FEATURES, "window": W, "horizon": 1, "roster_size": NR,
         "n_train_windows": int(len(ytr)), "n_test_windows": int(len(yte)),
         "n_train_series": len(eval_series)+len(train_only), "n_eval_series": len(eval_series),
         "n_train_only_series": len(train_only), "n_clients": len(tr_trusts), "n_test_clients": len(te_trusts),
         "train_windows_per_client_min": int(pd.Series(tr_owner).value_counts().min()),
         "feature_layout": f"{W} scaled lags + [sin,cos,covid,strike] + {NR} dept one-hot"}
(RESULTS/"dataset_stats.json").write_text(json.dumps(stats, indent=2))
print("leakage asserts: all 5 PASS")
print(f"X_train {Xtr.shape} | X_test {Xte.shape} | {N_FEATURES} features | {len(tr_trusts)} clients "
      f"| min train-windows/client {stats['train_windows_per_client_min']}")
print("GATE Step 2: PASS ->", RESULTS/"dataset_stats.json")

leakage asserts: all 5 PASS
X_train (108065, 33) | X_test (18191, 33) | 33 features | 117 clients | min train-windows/client 54
GATE Step 2: PASS -> /home/garima/fedOPD/results/dataset_stats.json


## Step 3 · Metric and the bar (R-01)  — the most important gate
sMAPE on **raw counts** (models inverse-transform first), computed **per series over its test months,
then aggregated**. R-01 is scored on **exactly the model's 18,191 (series,month) points** — the same
`te_df` rows that index `X_test`, not "all test months" — so every model number is comparable to it.
Two computations: (1) reproduce F-27 on the EDA's original scope (1,412 complete series, median 12.6/15.7,
±0.2) — this tests the metric, and is the **GATE**; (2) the operative bar on the frozen panel (R-01).

In [4]:
# Step 3 - metric (sMAPE on raw counts) + operative bar R-01, on EXACTLY the model's test points.
assert len(te_df) == len(Xte) == 18191, f"test set {len(te_df)} / X_test {len(Xte)} != 18191"  # index parity

def smape_raw(a, p):                                        # symmetric MAPE on raw counts
    a, p = np.asarray(a, float), np.asarray(p, float)
    return np.mean(2 * np.abs(p - a) / (np.abs(p) + np.abs(a) + 1e-6)) * 100

def score(pred_raw, meta=None):                            # per-series sMAPE -> macro-mean (headline) + median + vol-weighted
    meta = te_df if meta is None else meta
    df = meta.assign(pred=np.asarray(pred_raw, float))
    per = df.groupby(["branch", "dept"], sort=False).apply(
        lambda g: smape_raw(g.actual.values, g.pred.values), include_groups=False)
    mass = df.groupby(["branch", "dept"], sort=False).actual.sum()
    return {"macro_mean": float(per.mean()), "median": float(per.median()),
            "vol_weighted": float((per * mass).sum() / mass.sum()), "n_series": int(len(per))}

def to_raw(pred_scaled):                                    # models predict scaled -> inverse-transform before scoring
    return np.asarray(pred_scaled, float) * te_df["sd"].values + te_df["mean"].values

# naive t-1 and seasonal t-12 as RAW predecessors (present for every test point by window validity, D-02).
midx = {m: i for i, m in enumerate(months)}
tpos = te_df.month.map(lambda s: midx[pd.Timestamp(s)]).values
keys = list(zip(te_df.branch, te_df.dept))
raw_naive = np.array([dem.loc[k].values[t - 1]  for k, t in zip(keys, tpos)])
raw_seas  = np.array([dem.loc[k].values[t - 12] for k, t in zip(keys, tpos)])
assert not (np.isnan(raw_naive).any() or np.isnan(raw_seas).any()), "baseline predecessor missing"

R01_naive, R01_seas = score(raw_naive), score(raw_seas)
print("R-01 (panel, 18,191 pts) naive-t1  :", {k: round(v, 2) for k, v in R01_naive.items()})
print("R-01                     seasonal  :", {k: round(v, 2) for k, v in R01_seas.items()})

# GATE: reproduce F-27 on the EDA's original scope (1,412 complete series, MEDIAN) -> 12.6 / 15.7 +-0.2.
sd_all = canon.groupby(["branch", "dept"]).date.nunique()
nv, se = [], []
for (b, d) in sd_all.index[sd_all == 96]:
    v = canon[(canon.branch == b) & (canon.dept == d)].set_index("date").demand.sort_index().asfreq("MS").values
    if len(v) != 96 or np.isnan(v).any(): continue
    a = v[84:]; nv.append(smape_raw(a, v[83:95])); se.append(smape_raw(a, v[72:84]))
f27n, f27s = float(np.median(nv)), float(np.median(se))
gate = abs(f27n - 12.6) <= 0.2 and abs(f27s - 15.7) <= 0.2
print(f"F-27 reproduction (EDA scope, median): naive {f27n:.1f}% seasonal {f27s:.1f}%  (expect 12.6/15.7) -> {'PASS' if gate else 'FAIL'}")

json.dump({"R01_naive_t1": R01_naive, "R01_seasonal_t12": R01_seas,
           "f27_repro_median": {"naive": f27n, "seasonal": f27s},
           "n_test_points": int(len(te_df)),
           "bar_headline_macro_mean_pct": round(R01_naive["macro_mean"], 2),
           "bar_median_pct": round(R01_naive["median"], 2),
           "note": "R-01 = naive-t1 on the frozen 117-client panel. Headline metric is macro-mean sMAPE "
                   "(runbook); median (~12.9%) tracks the EDA F-27 12.6%. Models are scored identically."},
          open(str(RESULTS / "baselines.json"), "w"), indent=2)
assert gate, "F-27 not reproduced within tolerance - metric/aggregation disagrees with the EDA, STOP"
print("GATE Step 3: PASS | operative bar R-01 = naive-t1 macro-mean", round(R01_naive["macro_mean"], 2),
      "% (median", round(R01_naive["median"], 2), "%)")

R-01 (panel, 18,191 pts) naive-t1  : {'macro_mean': 16.34, 'median': 12.92, 'vol_weighted': 12.35, 'n_series': 1528}
R-01                     seasonal  : {'macro_mean': 23.26, 'median': 16.39, 'vol_weighted': 17.23, 'n_series': 1528}


F-27 reproduction (EDA scope, median): naive 12.6% seasonal 15.7%  (expect 12.6/15.7) -> PASS
GATE Step 3: PASS | operative bar R-01 = naive-t1 macro-mean 16.34 % (median 12.92 %)


## Step 4 · Forecaster + interface (modules)
**D-08a (module refactor, decided here as planned):** this notebook stays the narrative; the code
lives in `src/fedopd/{data,model,fed,evaluate,run}.py`, imported by both this notebook and the thin
CLI `python -m src.fedopd.run` (one process = one run = one `results/*.json`), so the Step-8 grid is
resumable — a kernel crash cannot kill a sweep. Model: **MLP 33→64→32→1, 4,289 params**, behind
`get_params / set_params / train_epochs(mu, global_params) / predict`. Two targets (D-08): `level`,
and **`delta`** (naive-anchored: zero-delta ≡ naive t-1). **GATE:** module/tensor parity + round-trip
+ overfit + determinism + delta anchor.

In [ ]:
# Step 4 - import the modules and re-run the gates (fast).
import sys; sys.path.insert(0, str(ROOT))
from src.fedopd import data as fdata, fed as ffed, evaluate as feval
from src.fedopd.model import Forecaster

dsm = fdata.build_all()                                     # module data plane
assert dsm["Xtr"].shape == tuple(Xtr.shape) and dsm["Xte"].shape == tuple(Xte.shape)
assert len(dsm["clients"]) == len(tr_trusts)                # 117 clients
bar_m = feval.naive_bar(dsm["te_df"]); assert abs(bar_m["macro_mean"] - R01_naive["macro_mean"]) < 1e-6

m = Forecaster(dsm["n_features"], "level", seed=0)
p0 = m.predict(dsm["Xte"][:256]); m.set_params(m.get_params())
assert (p0 == m.predict(dsm["Xte"][:256])).all()            # (a) round-trip bit-identical

key = dsm["te_df"].groupby(["branch", "dept"]).actual.sum().idxmax()
mask = dsm["owner"] == key[0]; Xs, ys = dsm["Xtr"][mask], dsm["ytr"][mask]
mo_ = Forecaster(dsm["n_features"], "level", 0, weight_decay=0.0)
mo_.train_epochs(Xs, ys, epochs=200, lr=3e-3, batch=32)
mse = float(np.mean((mo_.predict(Xs) - ys) ** 2)); assert mse < 0.01   # (b) overfit test

l1 = Forecaster(dsm["n_features"], "delta", 7).train_epochs(Xs, ys, 3, 1e-3, 32)
l2 = Forecaster(dsm["n_features"], "delta", 7).train_epochs(Xs, ys, 3, 1e-3, 32)
assert l1 == l2                                             # (c) determinism

zd = Forecaster(dsm["n_features"], "delta", 0)
zd.set_params([np.zeros_like(w) for w in zd.get_params()])
raw0 = zd.predict(dsm["Xte"]) * dsm["te_df"]["sd"].values + dsm["te_df"]["mean"].values
dev = np.abs(raw0 - dsm["te_df"]["naive_raw"].values).max(); assert dev < 1e-3  # (d) naive anchor
print(f"GATE Step 4: PASS  (params {sum(w.size for w in m.get_params())}, overfit MSE {mse:.5f}, "
      f"deterministic, delta-anchor max dev {dev:.1e} raw)")

## Steps 5–7 · Baselines, FedAvg, FedProx + personalisation (seed 0)
Runs live in `results/*.json` (via `run.py`; tmux `fedopd-step5`/`fedopd-step67`). Local-only gets the
pre-registered **fair fight** (train-tail early stopping, weight decay, small-client batch — H1);
personalisation fine-tunes the global model per client with tail-val early stopping (thin-client cap).
**D-08: target = delta** — beat `level` for both central (15.62 < 15.74) and local (16.99 < 18.38),
as F-27's persistence-domination predicted.

In [ ]:
# Steps 5-7 - seed-0 comparison table from the run JSONs (vs the R-01 bar on identical points).
import glob
def load(name):
    f = RESULTS / f"{name}_s0.json"
    return json.loads(f.read_text()) if f.exists() else None
rows = []
for name, label in [("central_delta", "centralised (upper bound)"), ("local_delta", "local-only"),
                    ("fedavg_ft3_delta", "fedavg"), ("fedprox_mu0.001_ft3_delta", "fedprox mu=.001"),
                    ("fedprox_mu0.01_ft3_delta", "fedprox mu=.01"), ("fedprox_mu0.1_ft3_delta", "fedprox mu=.1")]:
    r = load(name)
    if r is None: continue
    rows.append((label, r["metrics"]["macro_mean"], r["metrics"]["median"]))
    if "ft_metrics" in r:
        rows.append((label + " +ft3", r["ft_metrics"]["macro_mean"], r["ft_metrics"]["median"]))
tab = pd.DataFrame(rows, columns=["method", "macro_mean_sMAPE", "median_sMAPE"]).round(2)
bar = json.loads((RESULTS / "baselines.json").read_text())
print(f"bar R-01 naive-t1: macro {bar['R01_naive_t1']['macro_mean']:.2f} | median {bar['R01_naive_t1']['median']:.2f}")
print(tab.to_string(index=False))
print("\nreading: central beats the bar; +ft ties/near-ties it; every neural method beats naive on the")
print("MEDIAN by 1.6-1.9 pts - the macro-mean is dominated by the small/volatile-series tail (see Step 8).")

## Step 8 · Seeds, ablations, sensitivity, H1
Grid via `scripts/run_grid.sh` (tmux `fedopd-grid`, resumable — skips existing JSONs): headline
methods × 5 seeds; weighting {n, equal, sqrt}; window 12 vs 24; sensitivity = roster 17/18 (Q4:
Neurosurgery) / all-24 (Q5: X-codes) / panel 125/139/146 (Q1), each variant scored against **its own
recomputed bar**. **H1 (pre-registered):** the federation margin over local-only per client is
inversely related to the client's train-window count. **GATE:** rankings must survive seed noise.

In [ ]:
# Step 8a - seed table (mean +- std over 5 seeds) + ablations.
import glob, re
def collect(pattern):
    out = []
    for f in sorted(glob.glob(str(RESULTS / pattern))):
        r = json.loads(open(f).read())
        out.append(r)
    return out
def seed_stats(stem, key="metrics"):
    rs = [r for r in collect(f"{stem}_s*.json") if re.fullmatch(rf"{re.escape(stem)}_s\d+", r["name"] + "_s" + str(r["seed"])) or True]
    rs = [r for r in rs if r["name"] == stem]
    vals = [r[key]["macro_mean"] for r in rs if key in r]
    return (np.mean(vals), np.std(vals), len(vals)) if vals else (np.nan, np.nan, 0)
print(f"{'method':34s} {'macro mean+-std (n seeds)':>28s}")
for stem, key, label in [("local_delta", "metrics", "local-only"),
                         ("central_delta", "metrics", "centralised"),
                         ("fedavg_ft3_delta", "metrics", "fedavg"),
                         ("fedavg_ft3_delta", "ft_metrics", "fedavg +ft3"),
                         ("fedprox_mu0.001_ft3_delta", "metrics", "fedprox .001"),
                         ("fedprox_mu0.001_ft3_delta", "ft_metrics", "fedprox .001 +ft3")]:
    mu_, sd_, n_ = seed_stats(stem, key)
    print(f"{label:34s} {mu_:8.2f} +- {sd_:4.2f}  (n={n_})")
print(f"{'naive bar R-01':34s} {json.loads((RESULTS/'baselines.json').read_text())['R01_naive_t1']['macro_mean']:8.2f}")
print("\n-- ablations (seed 0) --")
for f, label in [("fedavg_ft3_delta_wequal_s0.json", "weighting equal"),
                 ("fedavg_ft3_delta_wsqrt_s0.json", "weighting sqrt(n)"),
                 ("fedavg_ft3_delta_win24_s0.json", "window 24 (fedavg)"),
                 ("central_delta_win24_s0.json", "window 24 (central)")]:
    fp = RESULTS / f
    if fp.exists():
        r = json.loads(fp.read_text())
        ft = f" | +ft {r['ft_metrics']['macro_mean']:.2f}" if "ft_metrics" in r else ""
        print(f"{label:22s} macro {r['metrics']['macro_mean']:.2f}{ft}  (bar for this variant {r['bar_R01']['macro_mean']:.2f})")

In [ ]:
# Step 8b - sensitivity (each variant vs ITS OWN recomputed naive bar) -> Q1/Q4/Q5 evidence.
print(f"{'variant':22s} {'central':>9s} {'fedavg':>8s} {'+ft':>7s} {'bar':>7s}   (macro-mean sMAPE)")
for tag, label in [("", "default (117x17)"), ("roster18", "incl. Neurosurgery"), ("all24", "all 24 depts"),
                   ("panel139", "panel >=90mo"), ("panel146", "panel >=84mo")]:
    sfx = f"_{tag}" if tag else ""
    c = RESULTS / f"central_delta{sfx}_s0.json"; a = RESULTS / f"fedavg_ft3_delta{sfx}_s0.json"
    if not (c.exists() and a.exists()): continue
    rc, ra = json.loads(c.read_text()), json.loads(a.read_text())
    print(f"{label:22s} {rc['metrics']['macro_mean']:9.2f} {ra['metrics']['macro_mean']:8.2f} "
          f"{ra['ft_metrics']['macro_mean']:7.2f} {ra['bar_R01']['macro_mean']:7.2f}"
          f"   [{ra['n_clients']} clients, {ra['metrics']['n_series']} series]")

In [ ]:
# Step 8c - H1 test (pre-registered): per-client margin (local - fedavg+ft) vs train-window count.
#   Positive margin = federation helped that client. H1 predicts margin shrinks as windows grow.
from scipy import stats as sstats
cd_ = fdata.per_client(dsm)
loc_models = ffed.train_local_only(cd_, dsm["n_features"], "delta", epochs=40, seed=0)
glob_, _ = ffed.run_federated(cd_, dsm["n_features"], "delta", rounds=40, seed=0)
pers_ = ffed.personalise(glob_, cd_, dsm["n_features"], "delta", ft_epochs=3, seed=0)
per_loc = feval.per_series_smape(dsm["te_df"], ffed.predict_per_client(loc_models, dsm["te_df"], dsm["Xte"]))
per_fed = feval.per_series_smape(dsm["te_df"], ffed.predict_per_client(pers_, dsm["te_df"], dsm["Xte"]))
cl_loc = per_loc.groupby("branch").smape.mean(); cl_fed = per_fed.groupby("branch").smape.mean()
nwin = pd.Series({c: len(y) for c, (X, y) in cd_.items()})
margin = (cl_loc - cl_fed).dropna()
rho, pval = sstats.spearmanr(nwin.loc[margin.index], margin)
q = pd.qcut(nwin.loc[margin.index], 4, labels=["thinnest", "q2", "q3", "largest"])
print("mean margin (local - fed+ft, pp) by client-size quartile:")
print(margin.groupby(q, observed=True).mean().round(2).to_string())
print(f"\nSpearman rho(train windows, margin) = {rho:.3f}  (p = {pval:.4f}, n = {len(margin)} clients)")
print("H1", "SUPPORTED" if (rho < 0 and pval < 0.05) else "NOT supported at 0.05", 
      "- negative rho = the data-poor gain most from federation")

## Step 9 · Flower parity
Same model + data under Flower (`flwr` 1.30): clients are `NumPyClient` wrappers, aggregation is
Flower's own `FedAvg.aggregate_fit`. Ray isn't installed, so transport is an in-process driver
(caveat logged); the FL math is Flower code. **GATE:** |flower − handwritten R-05| ≤ 0.3 sMAPE.

In [ ]:
# Step 9 - read the parity result (produced by scripts/flower_parity.py in tmux).
fp = RESULTS / "flower_parity_s0.json"
if fp.exists():
    r = json.loads(fp.read_text())
    print(f"flwr {r['flwr_version']}: macro {r['metrics']['macro_mean']:.2f} vs handwritten "
          f"{r['handwritten_R05']['macro_mean']:.2f} | gap {r['gap_macro']:.3f} -> GATE {r['gate']}")
    print("caveat:", r["caveat"])
else:
    print("flower_parity_s0.json not present - run: python scripts/flower_parity.py")